# 12. LLM 없이: 고정 + 문맥 추론 (12b)
사전학습 모델, LLM, LLM 생성 문장을 전혀 쓰지 않는다. 증강 없는 `outputs/` 의 01 과 v2 태거, 09 의 후보 캐시를 그대로 쓴다.

### 첫 시도(12)는 실패
빈칸 채우기 모델(가린 어절 맞히기)은 60 epoch 뒤에도 가린 글자 정확도 14% 에 그쳤고, 파이프라인 검증이 **0.9583** 으로 01(0.9834)보다 낮았다. 가린 글자에서만 학습 신호가 나와 1만 문장으로는 부족했고, 점수에 01 의 언어모델·사전 정보가 빠져 하한도 없었다.

### 12b 에서 바꾼 것
| | 12 | 12b |
|---|---|---|
| 신경망 모델 | 빈칸 채우기 (가린 글자만 학습) | 다음 글자 예측, 정방향·역방향 (`char_lm.py`, 모든 글자에서 학습) |
| 01 의 문자 5-gram | 사용 안 함 | 후보 주변 점수로 추가 (`w_kn`) |
| train 어절 빈도 | 선택 | 기본 탐색 (`w_freq`) |
| 하한 | 없음 | 01 유지 문턱(margin) 탐색 |

점수 = 채널 유사도 + w × 글자 언어모델 + w_kn × 문자 5-gram + w_freq × log(1 + train 빈도)

| 비교 기준 | 검증 F1 |
|---|---|
| 01 (LLM 없는 기존 최고) | 0.9834 |
| **신경망 없이** 5-gram + 빈도만 (로컬 CPU 에서 확인, w_kn 1.0~1.1, w_freq 1.0) | **0.9852~0.9853** |
| 같은 후보에서 최선을 고를 때 상한 | 0.9950 |
| 09 (같은 후보 + 미세조정 Qwen) | 0.9908 |

즉 신경망 모델이 전혀 도움이 안 돼도 0.985 는 나온다. 글자 언어모델이 그 위에 얼마를 더하는지가 이 노트북의 질문이다.

**GPU 런타임 (T4·L4 모두 가능), 첫 셀부터.**

## 0) 준비

In [ ]:
import os
os.chdir('/content')
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
DRIVE = '/content/drive/MyDrive/데이콘/난독화'
!mkdir -p /content/work && rsync -a --exclude '.playwright-mcp' --exclude '__pycache__' --exclude 'open.zip' --exclude 'v2_cache_*.pkl' --exclude 'models' "$DRIVE/" /content/work/
%cd /content/work
!ls outputs/val_pred.csv outputs/val_pred_tagger_v2_scratch.csv outputs/anchor_items_val_agree_v2_k20_b64_uo.pkl outputs/anchor_items_test_agree_v2_k20_b64_uo.pkl 2>&1; nproc; nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
def save_back():
    if not os.path.exists(DRIVE):
        drive.mount('/content/drive', force_remount=True)
    !rsync -a --update --exclude 'v2_cache_*.pkl' /content/work/outputs/ "$DRIVE/outputs/"
    print('saved outputs -> drive')

## 1) 글자 언어모델 학습 (train 90%, 30 epoch)
첫 줄에 **01 의 문자 5-gram 검증 손실**이 비교 기준으로 나온다. 매 epoch 의 '검증 손실(글자당) 정방향' 이 그 값보다 낮아져야 5-gram 보다 나은 언어모델이다.
검증 손실(정·역방향 평균)이 가장 낮은 시점을 `outputs/charlm_best.pt` 로 저장 (`*`). 중간부터 검증 손실이 올라가면 과적합이며, 저장된 최저 시점을 쓰므로 그대로 두면 된다. 기록: `outputs/charlm_runs_split.csv`.

In [ ]:
!python -u char_lm.py --epochs 30
save_back()

## 2) 검증: 가중치 탐색
1단계: w (글자 언어모델) × w_kn (5-gram) × w_freq (빈도) 조합. `w 0` 인 줄은 신경망 없는 결과다 (0.985 근처가 재현되면 정상).
2단계: 최고 조합에서 01 유지 문턱(margin).
결과: `outputs/val_pred_anchor_agree_v2_k20_b64_uo_12b.csv`, `best_anchor_..._12b.json`, `anchor_runs.csv` (name = 12b).

In [ ]:
!python -u anchor_lm12.py --mode val --unanchor_oov --cand_beam 64 --nbest 20 --weights 0,0.5,1.0,1.5 --kn_weights 0,0.5,1.0 --freq_weights 0,1.0 --margins 0.5,1,2
save_back()

## 3) 결과 비교
최고 조합이 탐색 범위 끝에 걸렸는지, 신경망 없는 최고(w = 0) 대비 얼마나 올랐는지 본다.

In [ ]:
import json
import pandas as pd
r = pd.read_csv('outputs/charlm_runs_split.csv')
r['val'] = (r.val_fwd + r.val_bwd) / 2
BEST_EP = int(r.loc[r.val.idxmin(), 'epoch'])
print('글자 언어모델: 최적 epoch', BEST_EP, '| 검증 손실 정방향', r.val_fwd.min(), '역방향', r.val_bwd.min())
a = pd.read_csv('outputs/anchor_runs.csv'); a['name'] = a['name'].astype(str)
g = a[a.name == '12b']
print('신경망 없이 (w = 0) 최고', g[g.w == 0].f1.max(), '| 전체 최고', g.f1.max())
print(g.sort_values('f1', ascending=False)[['w', 'w_kn', 'w_freq', 'margin', 'f1', 'sec']].head(10))
B = json.load(open('outputs/best_anchor_agree_v2_k20_b64_uo_12b.json', encoding='utf-8'))
v01 = pd.read_csv('outputs/val_pred.csv').set_index('ID')
v12 = pd.read_csv('outputs/val_pred_anchor_agree_v2_k20_b64_uo_12b.csv').set_index('ID')
d = v12.f1 - v01.f1.loc[v12.index]
print(f'12b F1 {v12.f1.mean():.4f} | 01 대비 좋아진 행 {(d>0).sum()} / 나빠진 행 {(d<0).sum()} | 01 0.9834, 09(LLM) 0.9908')

## 4) test 예측
최고 조합에서 글자 언어모델 가중치가 0 보다 크면 train 전체로 다시 학습(3 에서 구한 최적 epoch 수)한 뒤 적용한다. 0 이면 신경망 없이 바로 예측한다.
결과: `outputs/submission_anchor_agree_v2_k20_b64_uo_12b.csv`

In [ ]:
if B['w'] > 0:
    !python -u char_lm.py --full --epochs $BEST_EP
!python -u anchor_lm12.py --mode test --unanchor_oov --cand_beam 64 --nbest 20
save_back()

In [ ]:
!python -u anchor_lm12.py --mode val --unanchor_oov --cand_beam 64 --nbest 20 --weights 0.25,0.35,0.5,0.65,0.75 --kn_weights 0.25,0.5,0.75 --freq_weights 0 --margins 0 --name 12c
save_back()


### (선택) 런타임 해제

In [ ]:
from google.colab import runtime; runtime.unassign()